In [141]:
import torch
from torch import nn

from model_utils import load_transformer_model, load_silence_latent, load_encoder, decode_latent_and_save_audio, \
    load_finetuning_audio_latents, get_files_in_path_as_array

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model_dtype = torch.bfloat16
model_repo = "ACE-Step/acestep-v15-turbo-shift1"

cuda


In [3]:
vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device, model_dtype)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\clip\clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention


W0530 14:37:21.377000 19708 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [4]:
import torch, gc

path = "inputs/lora"
load_batch_size = 3 # due to VRAM constraints, can only load in limited batch sizes
train_data_limit = 10
files = get_files_in_path_as_array(path)
y = torch.Tensor().to(device).to(model_dtype)
with torch.no_grad():
    for start in range(0, min(len(files), train_data_limit), load_batch_size):
        end = min(start + load_batch_size, len(files))
        print(start, end)
        batch = load_finetuning_audio_latents(vae, files, device, model_dtype, start=start, end=end)
        y = torch.cat((y, batch), 0)
        del batch
        torch.cuda.empty_cache()
        gc.collect()
del vae

0 3
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
3 6
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
6 9
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
9 12
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
12 15
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
15 18
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
18 21
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])
21 24
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([2, 2880000])
torch.Size([3, 2, 2880000])


In [6]:
print(y.shape)

torch.Size([24, 64, 1500])


In [154]:
dit = load_transformer_model(model_repo, model_dtype, device)

In [145]:
for parameter in dit.parameters():
    parameter.requires_grad = False

In [146]:
rank = 8
for layer in dit.decoder.layers:
    layer.self_attn.q_proj.q_A = nn.Parameter(torch.randn(rank, layer.self_attn.q_proj.out_features, requires_grad=True, device=device, dtype=model_dtype))
    layer.self_attn.q_proj.q_B = nn.Parameter(torch.randn(layer.self_attn.q_proj.in_features, rank, requires_grad=True, device=device, dtype=model_dtype))
    layer.self_attn.v_proj.v_A = nn.Parameter(torch.randn(rank, layer.self_attn.v_proj.out_features, requires_grad=True, device=device, dtype=model_dtype))
    layer.self_attn.v_proj.v_B = nn.Parameter(torch.randn(layer.self_attn.v_proj.in_features, rank, requires_grad=True, device=device, dtype=model_dtype))


In [147]:
def lora_q_forward_hook(module, inputs, outputs):
    x = inputs[0]
    dW = module.q_B @ module.q_A
    return outputs + x @ dW

def lora_v_forward_hook(module, inputs, outputs):
    x = inputs[0]
    dW = module.v_B @ module.v_A
    return outputs + x @ dW

In [148]:
for layer in dit.decoder.layers:
    layer.self_attn.q_proj.register_forward_hook(lora_q_forward_hook)
    layer.self_attn.v_proj.register_forward_hook(lora_v_forward_hook)

In [155]:
# Generate music
silence_latent = load_silence_latent(model_repo, "silence_latent.pt", device, model_dtype)

text_hidden_states = torch.zeros(1, 77, 1024, dtype=model_dtype, device=device)
text_attention_mask = torch.zeros(text_hidden_states.shape[0], text_hidden_states.shape[1], dtype=torch.bool, device=device)
lyric_hidden_states = torch.zeros(1, 123, 1024, dtype=model_dtype, device=device)
lyric_attention_mask = torch.zeros(1, 123, dtype=torch.bool, device=device)

is_covers = torch.Tensor([False]).to(device)

seconds = 60
infer_steps = 50
frames_per_second = 25

seq_len = int(seconds * frames_per_second)

refer_audio_acoustic_hidden_states_packed = silence_latent[:, :, :1500].permute(0, 2, 1)
refer_audio_order_mask = torch.LongTensor([0]).to(device)

cur_chunk_mask = torch.ones(1, seq_len, 64, dtype=torch.bool, device=device)
cur_src_latents = silence_latent[:, :, :seq_len].permute(0, 2, 1)

outputs = dit.generate_audio(
    text_hidden_states=text_hidden_states,
    text_attention_mask=text_attention_mask,
    lyric_hidden_states=lyric_hidden_states,
    lyric_attention_mask=lyric_attention_mask,
    refer_audio_acoustic_hidden_states_packed=refer_audio_acoustic_hidden_states_packed,
    refer_audio_order_mask=refer_audio_order_mask,
    src_latents=cur_src_latents,
    chunk_masks=cur_chunk_mask,
    infer_steps=infer_steps,
    is_covers=is_covers,
    silence_latent=silence_latent,
    use_progress_bar=True,
    shift=1.0,

    repainting_start=torch.tensor([1.0]),
    repainting_end=torch.tensor([0.0]),
    audio_cover_strength=1.0,
    use_repainting=False
)
output_latents = outputs['target_latents'].transpose(1, 2).contiguous()

In [156]:
del dit
torch.cuda.empty_cache()
gc.collect()

22

In [157]:
print(output_latents.shape)

torch.Size([1, 64, 1500])


In [158]:
vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device, model_dtype)
with torch.no_grad():
    decode_latent_and_save_audio(output_latents, vae, "lora.wav")
del vae
torch.cuda.empty_cache()
gc.collect()

torch.Size([1, 2, 2880000])
(2880000, 2)


323